In [1]:
import pandas as pd
from pathlib import Path

source_file = Path("FY_2007_Tech_Doc.csv")

df = pd.read_csv(
    source_file,
    header=None,
    names=["line"],
    dtype=str,
    engine="python"
)

df["line"] = (
    df["line"]
    .fillna("")
    .str.replace("\x0c", "", regex=False)
    .str.strip()
)

sections = {
    "UNIT_Demo.csv": (
        "Unit Demographics and Sample Weights",
        "Unit Countable Income"
    ),
    "UNIT_Inc.csv": (
        "Unit Countable Income",
        "Unit Countable Assets"
    ),
    "UNIT_Assets.csv": (
        "Unit Countable Assets",
        "Unit Expenses and Deductions"
    ),
    "UNIT_ExDed.csv": (
        "Unit Expenses and Deductions",
        "Unit Benefits"
    ),
    "PERS_Char.csv": (
        "Person-Level Characteristics",
        "Person-Level Countable Income"
    ),
    "PERS_Inc.csv": (
        "Person-Level Countable Income",
        "Detailed Error Findings"
    ),
}

# This is the important part:
# Start near the Quick-Reference Codebook, not the Detailed Codebook.
quick_ref_start = df.index[
    df["line"].str.contains("Quick-Reference Codebook", case=False, regex=False)
][0]

def find_row(text, start=0):
    matches = df.index[
        df["line"].str.contains(text, case=False, regex=False)
    ]
    matches = [i for i in matches if i >= start]
    return matches[0]

for output_name, (start_text, end_text) in sections.items():
    start = find_row(start_text, quick_ref_start)
    end = find_row(end_text, start + 1)

    temp_df = df.loc[start:end-1, "line"].copy()

    # Keep only variable rows like: AGEi R Age
    temp_df = temp_df[
        temp_df.str.match(r"^[A-Za-z0-9_]+i?\s+[CR]\s+")
    ]

    # Keep only first word
    temp_df = temp_df.str.split().str[0]

    output_path = source_file.parent / output_name

    pd.DataFrame({"Table 1": temp_df}).to_csv(
        output_path,
        index=False
    )

    print(f"Created {output_name}: {len(temp_df)} rows")

Created UNIT_Demo.csv: 22 rows
Created UNIT_Inc.csv: 26 rows
Created UNIT_Assets.csv: 7 rows
Created UNIT_ExDed.csv: 27 rows
Created PERS_Char.csv: 14 rows
Created PERS_Inc.csv: 20 rows
